# Syntax: partial-matching beam search

This notebook illustrates `TreePathMatcher(method="beam")`, which now runs a beam search over valid partial matchings. A live beam state is a matched-pair prefix; each expansion appends a pair whose two coordinates are strict descendants of the previous pair. This is different from the exact DP table and from the older row-wise sparse-DP beam.

The examples use `TreeData` directly so the notebook is independent of `igraph` plotting setup.

In [ ]:
from pathlib import Path
import sys

# Allow running either from the repo root or from docs/.
_here = Path.cwd()
if (_here / "src").exists():
    sys.path.insert(0, str(_here / "src"))
elif (_here.parent / "src").exists():
    sys.path.insert(0, str(_here.parent / "src"))

import numpy as np
from path_matcher import TreeData, TreePathMatcher, align_trees_beam


def chain(labels):
    n = len(labels)
    parent = np.asarray([-1] + list(range(n - 1)), dtype=np.int32)
    return TreeData(parent=parent, label=list(labels), orig_index=np.arange(n, dtype=np.int32))


def tree(parent, labels):
    parent = np.asarray(parent, dtype=np.int32)
    return TreeData(parent=parent, label=list(labels), orig_index=np.arange(len(labels), dtype=np.int32))

## 1. Basic skipped-node example

The beam may jump from one matched pair to any descendant pair, so skipped vertices are implicit.

In [ ]:
G = chain("xAyBzC")
H = chain("ABC")

exact = TreePathMatcher(method="exact")
exact.fit(G, H)
print("exact:", exact.predict())

beam = TreePathMatcher(method="beam", beam_width=8, beam_expansion_width=8, seed=0)
beam.fit(G, H)
print("beam: ", beam.predict())

## 2. Beam parameters

`beam_width` controls how many partial matchings survive after each layer. `beam_expansion_width` controls how many candidate descendant pairs are generated per live state. Set `beam_expansion_width=None` only for tiny examples or debugging; it removes that cap for the generated positive label-pair buckets.

In [ ]:
small_beam = TreePathMatcher(
    method="beam",
    beam_width=2,
    beam_expansion_width=4,
    beam_random_fraction=0.0,
    seed=0,
)
small_beam.fit(G, H)
print(small_beam.predict())

## 3. Custom candidate heuristic

The default candidate heuristic combines match score, label rarity, gap penalties, and a small remaining-height estimate. You can replace the candidate ranking by passing `beam_candidate_heuristic_fn`. The callable receives a context object with fields such as `u`, `v`, `match_score`, `depth_gap`, `future_bound`, and `default_priority`.

In [ ]:
calls = []

def candidate_heuristic(ctx):
    calls.append((ctx.u, ctx.v))
    # Keep the default ranking, but very slightly favor candidates with more
    # possible continuation below them.
    return ctx.default_priority + 0.001 * ctx.future_bound

custom = TreePathMatcher(
    method="beam",
    beam_width=8,
    beam_expansion_width=8,
    beam_candidate_heuristic_fn=candidate_heuristic,
    seed=0,
)
custom.fit(G, H)
print(custom.predict())
print("heuristic calls:", len(calls))

## 4. Custom expansion rule

For full control, pass `beam_expansion_fn`. It receives the current terminal pair and returns feasible `(u, v)` pairs, or `(u, v, score)` triples. The implementation validates that returned pairs are strict descendants of the current terminal.

In [ ]:
G2 = chain("ABC")
H2 = chain("ABC")

def expansion(ctx):
    if ctx.last_u < 0:
        return [(0, 0)]
    u = ctx.last_u + 1
    v = ctx.last_v + 1
    if u < ctx.G.n and v < ctx.H.n:
        return [(u, v)]
    return []

res = align_trees_beam(G2, H2, beam_width=2, expansion_width=2, expansion_fn=expansion)
print(res.path_internal, res.score)

## 5. Note on farther lookahead

This implementation deliberately keeps lookahead simple: it uses remaining subtree height as a conservative future-score proxy. A stronger future heuristic could precompute coarse subtree-pair compatibility sketches or high-scoring chunk candidates and use those in `beam_priority_fn` or `beam_candidate_heuristic_fn`. That would move the method closer to A*-style guidance, but it is intentionally not part of this basic implementation.